# Simple Addition : additionner deux tableaux de 250 éléments sur le GPU

**But :** comprendre l'ordre des étapes qui entourent un calcul GPU, pour pouvoir le refaire seul.

## Le principe de l'ordre

Chaque étape ne mentionne que ce qui existe déjà. Trois parties :

1. **La mémoire et son contrat** : les données, les buffers, la description des emplacements, et leur branchement.
2. **Le programme** : on écrit le shader seulement maintenant, parce qu'il parle d'emplacements qui sont déjà décrits et branchés.
3. **L'exécution** : on construit la machine, on lance, on lit.

Deux règles imposent cet ordre :
- On ne peut pas créer un objet avant ceux qu'il référence.
- On ne peut pas exécuter avant d'avoir tout préparé, ni lire un résultat avant de l'avoir demandé.

## Pour retrouver l'ordre seul

1. Qu'est-ce que je veux calculer, avec quelles données ? (types et tailles)
2. Je me connecte au GPU, parce qu'il crée tout le reste.
3. Je réserve la mémoire et j'envoie les données.
4. Je décris les emplacements (le contrat).
5. Je branche les buffers sur ces emplacements.
6. J'écris le programme, qui utilise ces emplacements.
7. Je construis la machine (pipeline) à partir du programme et du contrat.
8. J'enregistre, j'envoie.
9. J'attends, je lis.

**✏️ Mes notes**



# Partie 1 : la mémoire et son contrat

## Étape 1 : les données sur le CPU

On prépare deux tableaux de 250 nombres. Tout le reste en dépend : la taille des buffers, le nombre de workgroups.

Le type est `float32` exprès : le shader lira des `f32`, soit quatre octets par nombre. NumPy utilise `float64` par défaut, ce qui donnerait des buffers deux fois trop grands et des résultats faux. **Le type de NumPy doit toujours correspondre à celui du shader.**

In [2]:
import numpy as np
import wgpu
from wgpu import BufferUsage, ShaderStage, BufferBindingType

N = 250
a = np.arange(N, dtype=np.float32)
b = np.arange(N, dtype=np.float32) * 10.0

**✏️ Mes notes**



## Étape 2 : ouvrir le canal avec le GPU

On demande un **adapter** (un GPU physique), puis un **device** (notre connexion de travail avec lui). Le device crée tous les objets suivants, donc rien ne peut exister avant lui.

In [3]:
adapter = wgpu.gpu.request_adapter_sync(power_preference="high-performance")
device = adapter.request_device_sync()

**✏️ Mes notes**



## Étape 3 : réserver et remplir la mémoire (buffers)

Trois buffers dans la VRAM :

- `buffer_a` et `buffer_b` : on réserve la place **et** on envoie les données en une seule fonction. Le GPU ne fera que les lire.
- `buffer_sum` : seulement réservé, avec la taille de `a`, parce que c'est le GPU qui le remplira. Il a `COPY_SRC` pour qu'on puisse le relire à la fin.

Cette étape ne dépend que du device et des tableaux.

In [4]:
buffer_a = device.create_buffer_with_data(data=a, usage=BufferUsage.STORAGE)
buffer_b = device.create_buffer_with_data(data=b, usage=BufferUsage.STORAGE)
buffer_sum = device.create_buffer(size=a.nbytes, usage=BufferUsage.STORAGE | BufferUsage.COPY_SRC)

**✏️ Mes notes**



## Étape 4 : décrire les emplacements (bind group layout)

On a trois buffers avec trois rôles : deux lus, un écrit. On décrit trois emplacements numérotés de 0 à 2, avec le type de chacun. Ce contrat ne dit pas **quel** buffer ira dans quel emplacement, seulement **quel genre** de chose l'emplacement accepte.

Il existe avant le programme parce que le programme devra s'y conformer, et parce que le compilateur a besoin de connaître cette forme avant de compiler.

- emplacement 0 : lecture seule (`read_only_storage`)
- emplacement 1 : lecture seule (`read_only_storage`)
- emplacement 2 : écriture (`storage`)

In [5]:
bind_group_layout = device.create_bind_group_layout(
    entries=[
        {"binding": 0, "visibility": ShaderStage.COMPUTE,
         "buffer": {"type": BufferBindingType.read_only_storage}},
        {"binding": 1, "visibility": ShaderStage.COMPUTE,
         "buffer": {"type": BufferBindingType.read_only_storage}},
        {"binding": 2, "visibility": ShaderStage.COMPUTE,
         "buffer": {"type": BufferBindingType.storage}},
    ]
)

**✏️ Mes notes**



## Étape 5 : brancher les buffers sur les emplacements (bind group)

Maintenant qu'on a les buffers (étape 3) et la description des emplacements (étape 4), on dit : emplacement 0 = `buffer_a`, emplacement 1 = `buffer_b`, emplacement 2 = `buffer_sum`.

Il référence le layout et les buffers, donc il ne peut venir avant aucun des deux.

In [6]:
bind_group = device.create_bind_group(
    layout=bind_group_layout,
    entries=[
        {"binding": 0, "resource": {"buffer": buffer_a, "offset": 0, "size": buffer_a.size}},
        {"binding": 1, "resource": {"buffer": buffer_b, "offset": 0, "size": buffer_b.size}},
        {"binding": 2, "resource": {"buffer": buffer_sum, "offset": 0, "size": buffer_sum.size}},
    ],
)

**✏️ Mes notes**



# Partie 2 : le programme

## Étape 6 : écrire le shader (WGSL)

On sait maintenant ce qui existe : trois emplacements, numérotés de 0 à 2, deux en lecture seule et un en écriture, tous dans le groupe 0. Le shader les déclare avec les mêmes numéros et les mêmes accès.

- Chaque ligne `@group(0) @binding(n) var<storage, ...>` correspond à l'emplacement `n` de l'étape 4. Le type `array<f32>` correspond au `float32` de l'étape 1.
- Chaque thread lit son numéro `i` et traite cette position.
- Le **garde-fou** est nécessaire : avec des workgroups de 64 threads, on lance 4 workgroups pour 250 éléments, donc 256 threads, dont 6 en trop. `arrayLength` donne la taille du tableau, qui vient de la taille du buffer.

In [7]:
shader_source = """
@group(0) @binding(0) var<storage, read> input_a: array<f32>;
@group(0) @binding(1) var<storage, read> input_b: array<f32>;
@group(0) @binding(2) var<storage, read_write> output_sum: array<f32>;

/* @brief Adds one element of input_a and input_b and writes it to output_sum at the same position.
   @param id Global invocation id of the thread: id.x is the index of the element it handles */
@compute @workgroup_size(64)
fn main(@builtin(global_invocation_id) id: vec3<u32>) {
    let i = id.x;
    if (i >= arrayLength(&input_a)) {
        return;
    }
    output_sum[i] = input_a[i] + input_b[i];
}
"""

**✏️ Mes notes**



## Étape 7 : charger le programme dans le GPU

On confie le texte du shader au device, qui le garde en mémoire. Il faut le device (étape 2) et le texte (étape 6). La vraie compilation vers le code machine se fait à l'étape suivante, quand le contrat est assemblé avec le programme.

In [8]:
shader_module = device.create_shader_module(code=shader_source)

**✏️ Mes notes**



# Partie 3 : l'exécution

## Étape 8 : assembler le pipeline

D'abord le **pipeline layout**, qui liste les tables d'emplacements utilisables (une seule ici, celle de l'étape 4). Puis le **pipeline**, qui combine le shader et ce layout. C'est à cet instant que le pilote **compile** et vérifie que shader et contrat sont d'accord.

Si ça plante ici, le shader et le contrat se contredisent (numéro, type ou accès).

In [9]:
pipeline_layout = device.create_pipeline_layout(bind_group_layouts=[bind_group_layout])
pipeline = device.create_compute_pipeline(
    layout=pipeline_layout,
    compute={"module": shader_module, "entry_point": "main"},
)

**✏️ Mes notes**



## Étape 9 : enregistrer les ordres, puis soumettre

On enregistre trois ordres dans la passe, **toujours dans cet ordre** : quel programme utiliser (pipeline), quelles données lui brancher (bind group), puis lancer. Le lancement vient en dernier parce qu'il a besoin des deux précédents.

- Nombre de workgroups = taille divisée par 64, **arrondie vers le haut**. Le 64 est celui de `@workgroup_size` dans le shader.
- `end` ferme la passe, `finish` ferme le lot de commandes, `submit` le remet au GPU. Chacun précède le suivant : on ne peut pas envoyer un lot qui n'est pas fini.

In [10]:
encoder = device.create_command_encoder()
compute_pass = encoder.begin_compute_pass()
compute_pass.set_pipeline(pipeline)
compute_pass.set_bind_group(0, bind_group)
compute_pass.dispatch_workgroups((N + 63) // 64)
compute_pass.end()
device.queue.submit([encoder.finish()])

**✏️ Mes notes**



## Étape 10 : récupérer et vérifier

La lecture vient **après** la soumission, sinon on lirait un buffer vide. Elle attend la fin du travail du GPU, puis ramène les octets, qu'on réinterprète en `float32`. On compare avec la somme calculée par NumPy.

**Ce que tu dois voir :** les premières valeurs `0, 11, 22, 33, 44`, puis `OK (250,)`.

In [11]:
result = np.frombuffer(device.queue.read_buffer(buffer_sum), dtype=np.float32)
print(result[:5], result[-3:])
print("OK" if np.allclose(result, a + b) else "WRONG", result.shape)

[ 0. 11. 22. 33. 44.] [2717. 2728. 2739.]
OK (250,)


**✏️ Mes notes**



## Les points de contact entre le shader et Python

Ce qui doit rester cohérent, et où se trouvent presque toutes les erreurs :

1. Les numéros de `@group` et de `@binding` correspondent aux entrées du layout (étape 4) et du bind group (étape 5).
2. Le mode d'accès (`read` ou `read_write`) correspond au type de buffer du layout.
3. Le type `f32` correspond au `dtype` NumPy `float32`.
4. La valeur de `@workgroup_size` correspond au calcul du dispatch.
5. Le nom `main` correspond à `entry_point`.
6. La longueur du tableau vient de la taille du buffer créé côté Python.

## Diagnostic : où chercher quand ça ne marche pas

- **Erreur à la création du pipeline** : le shader et le layout se contredisent.
- **Erreur à la relecture** : le buffer n'a pas le droit d'être copié vers l'extérieur (`COPY_SRC` manquant).
- **Rien ne change** : le `submit` manque, ou le dispatch lance zéro workgroup.
- **Résultat tout à zéro** : buffer non rempli, ou on relit le mauvais buffer.
- **Résultat faux sur les derniers éléments** : l'arrondi crée des threads en trop et le garde-fou manque.

**✏️ Mes notes**

